# SOXL发行人AUM/份额历史来源资格预检

只请求官方发行人产品页及其中一个明确同域CSV链接；保存精确响应与NAS哈希，核对2011–2022日级资产/份额及历史首次公布资格。不联系供应商，不读市场结果。

In [1]:
import csv,datetime as dt,hashlib,io,json,shlex,subprocess,tarfile,tempfile
from html.parser import HTMLParser
from pathlib import Path
from urllib.parse import urljoin,urlparse
from urllib.request import Request,build_opener,HTTPRedirectHandler
from urllib.error import HTTPError
from src.data.nas_market_data import nas_api_call,_settings,_ssh
root=Path.cwd();card=root/'research/stage1_soxl_rebalance_aum_source_triage_20261024'
plan=json.loads((card/'plan.json').read_text());assert plan['frozen'] and not (card/'run_state.json').exists()
state_path=card/'run_state.json';state={'status':'started','plan_sha256':hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),'GETs_started':0};state_path.write_text(json.dumps(state,indent=2)+'\n')
raw=root/'data/snapshots/stage1_soxl_rebalance_aum_source_triage_20261024';raw.mkdir(parents=True,exist_ok=True)
class NoRedirect(HTTPRedirectHandler):
 def redirect_request(self,req,fp,code,msg,headers,newurl):return None
class Links(HTMLParser):
 def __init__(self):super().__init__();self.href=None;self.text=[];self.links=[]
 def handle_starttag(self,tag,attrs):
  if tag.lower()=='a':self.href=dict(attrs).get('href');self.text=[]
 def handle_data(self,data):
  if self.href is not None:self.text.append(data)
 def handle_endtag(self,tag):
  if tag.lower()=='a' and self.href is not None:
   self.links.append({'text':' '.join(''.join(self.text).split()),'href':self.href});self.href=None;self.text=[]
opener=build_opener(NoRedirect)
def fetch(url,label):
 host=urlparse(url).hostname;assert host in plan['allowed_hosts']
 now=dt.datetime.now(dt.timezone.utc);req=Request(url,headers={'User-Agent':'Mozilla/5.0 source qualification research','Accept':'text/html,text/csv,*/*'})
 status=None;headers={};body=b'';error=None
 try:
  with opener.open(req,timeout=30) as response:
   status=response.status;headers={k.lower():v for k,v in response.headers.items()};body=response.read(plan['request_budget']['max_bytes_per_response']+1)
 except HTTPError as exc:
  status=exc.code;headers={k.lower():v for k,v in exc.headers.items()};body=exc.read(plan['request_budget']['max_bytes_per_response']+1);error='HTTPError '+str(exc.code)
 except Exception as exc:error=type(exc).__name__+': '+str(exc)
 path=raw/(label+'.bin')
 if body:path.write_bytes(body)
 receipt={'label':label,'url':url,'requested_at_utc':now.isoformat(),'received_at_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
          'http_status':status,'content_type':headers.get('content-type'),'response_date':headers.get('date'),
          'cache_control':headers.get('cache-control'),'last_modified':headers.get('last-modified'),
          'bytes':len(body),'sha256':hashlib.sha256(body).hexdigest() if body else None,'relative_path':str(path.relative_to(root)) if body else None,'error':error}
 if len(body)>plan['request_budget']['max_bytes_per_response']:receipt['error']='response exceeded frozen max bytes'
 return receipt,body
requests=[];state.update(GETs_started=1);state_path.write_text(json.dumps(state,indent=2)+'\n')
page,body=fetch(plan['official_page'],'issuer_page');requests.append(page)
links=[];terms={};csv_result=None
if page['http_status']==200 and body and not page['error'] and str(page['content_type']).lower().startswith('text/html'):
 text=body.decode('utf-8',errors='replace');parser=Links();parser.feed(text)
 links=[{'text':z['text'],'url':urljoin(plan['official_page'],z['href'])} for z in parser.links if any(term in (z['text']+' '+z['href']).lower() for term in ['holdings','assets','shares outstanding','aum','flow','history','csv'])]
 terms={term:text.lower().count(term) for term in ['shares outstanding','assets under management','net assets','historical assets','daily fund holdings','daily fund flows']}
 choices=[z for z in links if 'daily fund holdings' in z['text'].lower() and urlparse(z['url']).hostname in plan['allowed_hosts'] and ('.csv' in urlparse(z['url']).path.lower())]
 if len(choices)==1:
  state.update(GETs_started=2);state_path.write_text(json.dumps(state,indent=2)+'\n')
  csv_receipt,csv_body=fetch(choices[0]['url'],'linked_holdings_csv');requests.append(csv_receipt)
  if csv_receipt['http_status']==200 and csv_body and not csv_receipt['error']:
   rows=list(csv.reader(io.StringIO(csv_body.decode('utf-8',errors='replace'))))
   csv_result={'rows':len(rows),'header':rows[0][:30] if rows else [],'first_rows':rows[1:4] if len(rows)>1 else [],'last_rows':rows[-3:] if len(rows)>4 else [],'date_like_rows':sum(any('20' in cell and ('-' in cell or '/' in cell) for cell in row) for row in rows[1:])}
result={'status':'source_only_needs_historical_daily_AUM_shares_and_first_public_vintages','plan_sha256':hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),
        'official_requests':requests,'relevant_official_page_links':links[:40],'official_page_term_counts':terms,'linked_csv_inspection':csv_result,
        'daily_2011_2022_AUM_shares_qualified':False,'historical_first_public_vintages_qualified':False,
        'new_market_GETs':len(requests),'future_market_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0,'vendor_contacted':False,
        'limits':['A current issuer page or holdings CSV does not by itself prove a 2011-2022 daily fund-size/share series or its first publication time','No third-party AUM estimate or inferred ETF rebalancing flow was used']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
if any(z['relative_path'] for z in requests):
 with tempfile.TemporaryDirectory(prefix='soxl-aum-source-archive-') as tmp:
  archive=Path(tmp)/'source.tar.gz'
  with tarfile.open(archive,'w:gz') as tar:
   for z in requests:
    if z['relative_path']:
     path=root/z['relative_path'];info=tar.gettarinfo(str(path),arcname=Path(z['relative_path']).name);info.mtime=0
     with path.open('rb') as stream:tar.addfile(info,stream)
   payload=json.dumps({'plan_sha256':result['plan_sha256'],'requests':requests},ensure_ascii=False,sort_keys=True).encode()
   info=tarfile.TarInfo('receipt.json');info.size=len(payload);info.mtime=0;tar.addfile(info,io.BytesIO(payload))
  digest=hashlib.sha256(archive.read_bytes()).hexdigest();settings=_settings(root)
  nas=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
  assert nas['sha256']==digest and nas['size_bytes']==archive.stat().st_size
  remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
  assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
  restored=Path(tmp)/'restored.tar.gz'
  with restored.open('wb') as out:
   proc=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert proc.returncode==0 and hashlib.sha256(restored.read_bytes()).hexdigest()==digest
  archive_receipt={'sha256':digest,'size_bytes':archive.stat().st_size,'nas_path':remote,'remote_hash_verified':True,'scoped_readback_verified':True,'raw_member_count':sum(bool(z['relative_path']) for z in requests)}
  (card/'nas_source_archive_receipt.json').write_text(json.dumps(archive_receipt,indent=2)+'\n')
state.update(status='completed',GETs_completed=len(requests));state_path.write_text(json.dumps(state,indent=2)+'\n')
print({'status':result['status'],'requests':[(z['label'],z['http_status'],z['bytes']) for z in requests],'link_count':len(links),'linked_csv':csv_result is not None})


{'status': 'source_only_needs_historical_daily_AUM_shares_and_first_public_vintages', 'requests': [('issuer_page', 200, 1204411)], 'link_count': 6, 'linked_csv': False}


## 中文结论

# SOXL实际再平衡需求的发行人来源仍缺日级历史与公布时钟

按冻结计划只请求一次[Direxion官方SOXL/SOXS产品页](https://www.direxion.com/product/daily-semiconductor-bull-bear-3x-etfs)，收到HTTP 200、**1,204,411字节**HTML；UTC请求/响应时刻、响应头和SHA-256记录于`result.json`。原HTML压缩包SHA-256 `0ffa12b3f888d6df690b64d3ccdc1430ff182c3ee3f095dac55fb6ce0756ade4` 已上传Yanxuan NAS、核对远端哈希并限定恢复。未向供应商联系、购买或使用第三方估计；0市场结果、拟合、账户。

该页面当前呈现NAV/市场价与“Daily Fund Holdings (csv)”字样，但静态HTML中的该链接指回**同一产品页**；固定计划只允许跟随一个明确的同域直接CSV，故未执行第二次请求。页面文本未给出“shares outstanding”或“AUM”字段，也没有2011–2022逐日资产/份额表及各日原始首次公布版本。**这只是对本次官方产品页入口的限定结论**，不证明发行人其他动态接口或档案不存在。

按来源门决定 `source_only_needs_historical_daily_SOXL_AUM_shares_and_first_public_vintages`。没有日级基金规模与申赎份额，就不能把前卡的尾盘SOXL−3×SOXX价格残差冒充实际杠杆ETF再平衡需求，也不能重开已失败的固定局部信息门。若后续发现官方历史接口，需新来源计划核对覆盖、单位/拆分、修订及在下一Stage1决策前的可用时间，再冻结不同的因子与经济接收器。

首次执行在HTTP调用前因状态变量类型错误停止，`attempt1_preflight_failure.json`记录0 GET；机械修复后仅完成1次官方请求，没有追加预算或备用站点。SOXX稳定净超额、2/5bp和独立前向均未达到。
